# Multiclass Classification: Iris Data Set

(Adapted from Professor Kyle Whynott's CPSC 483 course material)

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

## 1. Frame the Problem

![image.png](attachment:image.png)


### Dataset: 150 iris flowers from three species: Iris setosa, Iris versicolor, and Iris virginica.
### Features: Sepal length, sepal width, petal length, and petal width (4 numerical features).

## 2. Get the Data

In [ ]:
data = load_iris()

In [ ]:
#display(data)
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

### Quick Look at Data

In [ ]:
display(X)
display(y.value_counts())

In [ ]:
X.info()
y.info()

### Create the Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.3,
    stratify=y,
    random_state=42
)

## 3. Exploratory Data Analysis (EDA)

### Visualize the Feature Distribution (something new in this project)
#### The diagonal shows the distribution of each feature for each species.
Example: Petal length diagonal

Setosa has very small values

Versicolor medium

Virginica large
#### This means: Petal length is very discriminative.

#### Each off-diagonal panel shows the relationship between two features.

Example:

Petal length vs Petal width:

Very clear separation between species

Almost linearly separable

#### That means: These two features are very powerful for classification.

In [ ]:
sns.pairplot(

    pd.concat([X_train, y_train], axis=1),
    hue="target"
)
plt.show()

In [ ]:
X_train.hist(figsize=(24,16)) # this is the one we did in the breast cancer prediction project
plt.suptitle("Feature Distributions")
plt.show()

### Correlation HeatMap

In [ ]:
sns.heatmap(X_train.corr(),cmap="YlGnBu", linewidths=1, xticklabels=True, yticklabels=True)

## 4. Prepare the Data and Model for ML

### Data Cleaning

In [ ]:
X_train.dropna(axis=0, how="all", inplace=True)
X_train.dropna(axis=1, how="all", inplace=True)
X_test.dropna(axis=0, how="all", inplace=True)
X_test.dropna(axis=1, how="all", inplace=True)

y_train.dropna(axis=0, how="all", inplace=True)
y_test.dropna(axis=0, how="all", inplace=True)

In [ ]:
print(f" X_train: {len(X_train)}")
print(f" y_train: {len(y_train)}")
print(f" X_test: {len(X_test)}")
print(f" y_test: {len(y_test)}")

### Build the Data Pipeline

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

features = X_train.select_dtypes(include=["float64"]).columns
pipeline = Pipeline([
    ("scaler", StandardScaler())
])

preprocessing = ColumnTransformer([
    ("num",pipeline, features)
])

### Create a Baseline Model

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

baseline = Pipeline([
    ("prep", preprocessing),
    ("model", DummyClassifier(strategy="stratified"))
])

# Stratified K-Fold Cross Validation
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
baseline_scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,
    verbose=3
)
baseline_scores.mean()

## 5a. Select and Train a Model - Using OVR (One Versus Rest)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier #use OVR as an adaptor for LogisticRegression to handle multiclass classification

iris_classifier_ovr = make_pipeline(
    preprocessing,
    OneVsRestClassifier(# put LogisticRegression into OneVsRestClassifier
        LogisticRegression(
            class_weight="balanced",
            solver="newton-cg", # in the breast cancer classfication project, we used 'lbfgs', but here we use 'newton-cg'
            # for details of these hyperparameters, please refer to the documentation:
            # https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
            random_state=42
        )
    )
)
iris_classifier_ovr.fit(X_train, y_train)

### Re-evaluate with K-Fold Cross Validation (Stratified)

In [ ]:
ic = cross_val_score(
    iris_classifier_ovr,
    X_train,
    y_train,
    scoring="accuracy",
    cv=skf
)

ic.mean()

## 5b. Select and Train a Model - Using SoftMax

In [ ]:
from sklearn.linear_model import LogisticRegression
#by default, LogisticRegression handles multiclass classification using the Softmax function
### To do: apply your model here
### ??????
iris_classifier.fit(X_train, y_train)

### Re-evaluate with K-Fold Cross Validation (Stratified)

In [ ]:
ic = cross_val_score(
    iris_classifier,
    X_train,
    y_train,
    scoring="accuracy", # use accuracy as the evaluation metric for multiclass classification
    cv=skf
)

ic.mean()

#### Average accuracy in K-Fold Cross Validation:
#### OvR vs Softmax
#### 0.89   0.96
#### we select Softmax in next step to fine tune the model


## 6. Fine Tune the Model

In [ ]:
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("logistic_regressor", LogisticRegression(class_weight="balanced", solver="newton-cg", random_state=42))
     ])

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {

    # Random Forest hyperparameters
    "logistic_regressor__l1_ratio": [0],
    "logistic_regressor__fit_intercept": [True, False],
    "logistic_regressor__intercept_scaling": [0.5, 1, 2],
    "logistic_regressor__max_iter": [1000, 2500, 5000],
}

# Initialize the GridSearchCV object
grid_search = GridSearchCV(full_pipeline, param_grid, cv=skf, scoring='accuracy', n_jobs=-1, verbose=3)

# Fit the grid search to the data
grid_search.fit(X_train, y_train)

# Print the best parameters and the best score achieved during the grid search
print("\n\nBest parameters:", grid_search.best_params_)
print("\nBest cross-validation score:", grid_search.best_score_)

## 7. Create, Evaluate, and Save the Finalized Model

In [ ]:
ic = make_pipeline(preprocessing, LogisticRegression(class_weight="balanced",fit_intercept=True,intercept_scaling=0.5,l1_ratio=0,max_iter=1000,solver="newton-cg", random_state=42))
ic.fit(X_train,y_train)

In [ ]:
# display the coefficients of the trained logistic regression model
# we will see 12 coefficient this time , becuase we have 4 features and 3 classes, so we will have 4*3=12 coefficients in total
ic[1].coef_

### Evaluate on the Test Set

In [ ]:
# Hard predictions
y_test_pred = ic.predict(X_test)

# Probabilities (needed for ranking metrics)
y_test_proba = ic.predict_proba(X_test)

In [ ]:
print(y_test_pred)

In [ ]:
print(y_test_proba)

### Let's manually compute Softmax before using sklearn.

#### We will :

Get raw scores (logits)

Apply Softmax manually

Verify probabilities sum to 1

In [ ]:
#check the raw socre value before it goes to the softmax function
logits = ic.decision_function(X_test.iloc[0:1]) # we need to use X_test[0:1] instead of X_test[0] because the decision_function method expects a 2D array as input, not a 1D array
display(X_test.iloc[0]) #1D array
display(X_test.iloc[0:1])#2D array

display(logits)

z = logits[0]
### To do: check if the softmax probabilities sum to 1
display('here are scores  for the first test sample')
display(z)
#????

display('here are softmax probabilities for the first test sample')
#?????
# display("Sum of softmax probabilities:", ?????))


#### The confusion matrix for multiclass classfication is as follows
![image.png](attachment:image.png)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
class_names = [str(cls) for cls in sorted(y_train.unique())]
conf_matrix = confusion_matrix(y_test, y_test_pred)
class_report = classification_report(y_test, y_test_pred, target_names=class_names)
print('Confusion Matrix for class name :\n', conf_matrix)
print('Classification Report:\n', class_report)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Final Confusion Matrix')
plt.show()

In [ ]:
class_report = classification_report(y_test, y_test_pred, target_names=class_names, output_dict=True)
df_final_report = pd.DataFrame(class_report).transpose()

plt.figure(figsize=(10, 5))
sns.heatmap(df_final_report.iloc[:-1, :].drop(columns=['support']), annot=True, cmap='Blues', cbar=False)
plt.title('Classification Report')
plt.show()

In [ ]:
import joblib
joblib.dump(ic, "./models/iris.pkl")


# 🔄 Future Usage: Loading a Saved Model

In this section, we simulate working in a **new notebook in the future**.
We will load the previously saved pipeline model (`iris.pkl`),
which already includes preprocessing and the trained Logistic Regression model.


In [ ]:

# ================================
# FUTURE USE: Load Saved Model
# ================================

import joblib

# Load the saved pipeline model (preprocessing + logistic regression)
model = joblib.load("./models/iris.pkl")

print("Model loaded successfully!")
print("Model type:", type(model))



# 📊 Step 1: Load the Iris Dataset Again

Even though the model is already trained, we still need input data.
Here we reload the Iris dataset to simulate new incoming data.


In [ ]:

# ================================
# Load Iris Dataset Again
# ================================

from sklearn.datasets import load_iris

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
target_names = iris.target_names


print("Dataset loaded.")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Class names:", target_names)



# ✂️ Step 2: Create a Test Split

We create a test set to evaluate how well the loaded model performs.
Using the same `random_state=42` ensures reproducibility.


In [ ]:

# ================================
# Create Test Split (Future Testing)
# ================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Test set shape:", X_test.shape)



# 📈 Step 3: Evaluate the Loaded Model

Now we use the loaded model to:
- Predict on the test data
- Compute accuracy
- Display the confusion matrix
- Show the classification report (precision, recall, F1-score)


In [ ]:

# ================================
# Evaluate Loaded Model
# ================================

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=target_names))
